# Funciones de Utilidad y Modelos de TorchVision

En este laboratorio, pasarás de la teoría a la práctica aprovechando el poder de la librería `torchvision`. En el panorama actual de la IA, el desafío no es solo construir modelos desde cero, sino desplegar soluciones potentes, precisas e interpretables de manera eficiente. Aquí es donde `torchvision` se convierte en una pieza indispensable de tu kit de herramientas. Este laboratorio está diseñado para llevarte más allá de la teoría y sumergirte en el flujo de trabajo profesional utilizando modelos preentrenados de última generación para resolver problemas del mundo real.

En este laboratorio, verás cómo:

* Utilizar las utilidades de `torchvision` para visualizar las predicciones del modelo con **cajas delimitadoras (bounding boxes)** y **máscaras de segmentación**.

* Inspeccionar modelos preentrenados para comprender su arquitectura e identificar sus clases de salida.

* Realizar **inferencia** con modelos preentrenados para clasificación, segmentación y detección de objetos.



El laboratorio culmina con una sección práctica de "¡Inténtalo tú mismo!" donde ejecutarás estos modelos en tus propias imágenes cargadas, consolidando tu aprendizaje y preparándote para proyectos del mundo real.

## Imports

In [ ]:
from pprint import pprint

import torch
import torchvision.models as tv_models
import torchvision.transforms as transforms
import torchvision.utils as vutils
from IPython.display import Image as DisplayImage
from PIL import Image
from torchvision.io import decode_image

import helper_utils

## Utilidades de TorchVision para la Anotación de Imágenes

La salida de un modelo, ya sea un conjunto de coordenadas o un mapa denso de puntuaciones de clase, es información abstracta hasta que puedes verla. ¿Cómo sabes si tu detector de objetos funciona correctamente? Dibujas sus predicciones directamente sobre la imagen. Este acto de visualización es un paso fundamental en cualquier proyecto de visión por computadora, transformando datos tensoriales brutos en resultados claros y comprensibles para el ser humano.

En esta sección, verás en acción dos de las utilidades más prácticas de TorchVision:

* `draw_bounding_boxes` para colocar cajas alrededor de los objetos que un modelo ha identificado.
* `draw_segmentation_masks` para superponer máscaras detalladas a nivel de píxel sobre objetos específicos.



Estas funciones son tus herramientas principales para depurar, evaluar el rendimiento del modelo y crear demostraciones visuales convincentes de tu trabajo.

### Dibujando Cajas Delimitadoras (Bounding Boxes)

La tarea principal de un modelo de detección de objetos es encontrar objetos y señalar sus ubicaciones. La forma clásica y más directa de visualizar estos datos de ubicación es mediante el dibujo de **bounding boxes**, rectángulos simples que enmarcan cada objeto detectado.

Utilizarás la función `draw_bounding_boxes` para realizar esta tarea. Toma tu imagen original y un conjunto de coordenadas de cajas para crear instantáneamente una representación visual clara de los hallazgos de tu modelo. Esta es la misma técnica utilizada en aplicaciones que van desde coches autónomos que identifican peatones hasta sistemas de pago automatizados que registran productos.


* Define coordenadas fijas (hardcoded) para dos cajas delimitadoras.
    * En escenarios prácticos, estas coordenadas son devueltas por un modelo de detección.
* Define una lista de cadenas, `labels`, que correspondan a cada caja delimitadora.
* Usa <code>[draw_bounding_boxes()](https://docs.pytorch.org/vision/main/generated/torchvision.utils.draw_bounding_boxes.html)</code> para dibujar las cajas definidas.
    * `image`: El tensor `uint8` o `float` de forma (`C, H, W`) sobre el cual se dibujarán las cajas.
    * `boxes`: Un tensor de forma (`N, 4`), donde `N` es el número de cajas y `4` corresponde a las coordenadas (`xmin, ymin, xmax, ymax`).
    * `labels`: Lista de cadenas para mostrar en cada caja delimitadora.
    * `colors`: Lista de colores utilizados para representar cada etiqueta u objeto. Cada color corresponde a un objeto detectado específico y se usa al dibujar su caja.
    * `width=3`: Establece el ancho de línea en píxeles para la caja delimitadora en 3.

In [ ]:
# Cargar la imagen
image = decode_image('./images/dog1.jpg')

# Bounding boxes de muestra, donde cada caja está en formato (xmin, ymin, xmax, ymax)
# La primera caja se ajusta para rodear a todo el perro, la otra para rodear el ojo izquierdo
boxes = torch.tensor([[140, 30, 375, 315], [200, 70, 230, 110]], dtype=torch.float)

# Etiquetas correspondientes para los objetos detectados
labels = ["dog", "eye"]

# Dibujar las cajas sobre la imagen
result = vutils.draw_bounding_boxes(image=image, 
                                    boxes=boxes, 
                                    labels=labels,           # Esto es opcional
                                    colors=["red", "blue"],  # Esto es opcional. Por defecto, se generan colores aleatorios.
                                    width=3                  # Esto es opcional. El valor por defecto es width=1
                                   )

# Mostrar el resultado
helper_utils.display_images(processed_image=result, figsize=(10, 10))

### Dibujando Máscaras de Segmentación

Mientras que las cajas delimitadoras te dicen *dónde* está un objeto, a veces necesitas conocer su **forma** exacta. Para tareas que requieren un mayor nivel de precisión, debes ir más allá de las cajas y clasificar cada píxel de un objeto.

Este es el trabajo de la segmentación de imágenes, que genera una **máscara de segmentación**, una superposición perfecta a nivel de píxel que resalta la silueta precisa de un objeto. Esta comprensión detallada es indispensable en aplicaciones avanzadas. Por ejemplo, un coche autónomo utiliza la segmentación para entender el límite exacto de la carretera, y los modelos de imágenes médicas dependen de ella para delinear tumores con precisión clínica.

Utilizarás la función `draw_segmentation_masks` para dar vida a estas complejas predicciones sobre una imagen.

* Carga la `object_mask` precalculada desde un archivo.
    * Para fines de demostración, esta máscara fue precalculada para funcionar solo con `dog1.jpg`.
    * En escenarios prácticos, las máscaras son devueltas por un modelo de segmentación.
* Usa <code>[draw_segmentation_masks()](https://docs.pytorch.org/vision/main/generated/torchvision.utils.draw_segmentation_masks.html)</code> para dibujar la máscara de segmentación definida.
    * `image`: El tensor `uint8` o `float` de forma (`3, H, W`) sobre el cual se dibujarán las máscaras.
    * `masks`: Un tensor **booleano** de forma (`num_masks, H, W`) o (`H, W`), donde `num_masks` es el número de máscaras y los valores `True` indican los píxeles que se deben colorear.
    * `alpha`: Un **float** entre `0` (completamente transparente) y `1` (completamente opaco) que controla la transparencia de la máscara.
    * `colors`: Lista de colores utilizados para representar cada objeto. Cada color corresponde a un objeto detectado y se usa al dibujar su máscara.

In [ ]:
# Load the pre-saved segmentation mask
mask_filename = "dog_segmentation_mask.pt"
loaded_object_mask = torch.load(mask_filename)

# Make it (1, H, W)
object_mask = loaded_object_mask.unsqueeze(0)


# Draw segmentation mask on the image
result  = vutils.draw_segmentation_masks(image=image,
                                         masks=object_mask,
                                         alpha=0.5,          # This is optional. The default is alpha=0.8
                                         colors=["blue"]     # This is optional. By default, random colors are generated for each mask.
                                        )

# Display the result
helper_utils.display_images(processed_image=result, figsize=(10, 10))

## Tu Kit de Herramientas de Modelos Preentrenados

Entrenar un modelo importante de visión por computadora desde cero es un esfuerzo monumental que requiere conjuntos de datos masivos y semanas de tiempo de GPU. TorchVision te ofrece un atajo potente: una librería profesional de **modelos preentrenados**.

Piensa en ellos como modelos expertos que ya han sido entrenados en datasets gigantescos como ImageNet o COCO. Vienen con una comprensión integrada de las características visuales, desde texturas simples hasta objetos complejos como rostros y vehículos. Ahora puedes usar este conocimiento experto como un punto de partida poderoso para tus propios proyectos.



**Un Recorrido por las Arquitecturas Disponibles**

TorchVision organiza sus modelos según la tarea principal para la que fueron diseñados. Estas son las categorías principales:

* **Clasificación de Imágenes**: Responde a la pregunta básica: "¿Cuál es el sujeto principal de esta imagen?"
    * **Modelos**: [ResNet](https://docs.pytorch.org/vision/main/models/resnet.html), [VGG](https://docs.pytorch.org/vision/main/models/vgg.html), [AlexNet](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.alexnet.html), [SqueezeNet](https://docs.pytorch.org/vision/main/models/squeezenet.html), [MobileNetV3](https://docs.pytorch.org/vision/main/models/mobilenetv3.html), [DenseNet](https://docs.pytorch.org/vision/main/models/densenet.html)
* **Segmentación de Imágenes**: Va más allá para preguntar: "¿Cuál es la forma exacta, píxel por píxel, de cada objeto?"
    * **Modelos**: [FCN](https://docs.pytorch.org/vision/main/models/fcn.html), [DeepLabV3](https://docs.pytorch.org/vision/main/models/deeplabv3.html)
* **Detección de Objetos**: Encuentra todos los objetos reconocibles en una imagen, dibuja una caja alrededor de cada uno y los clasifica.
    * **Modelos**: [Faster R-CNN](https://docs.pytorch.org/vision/main/models/faster_rcnn.html), [RetinaNet](https://docs.pytorch.org/vision/main/models/retinanet.html), [SSD](https://docs.pytorch.org/vision/main/models/ssd.html)
* **Clasificación de Video**: Comprende la acción y el movimiento clasificando clips de video completos.
    * **Modelos**: [R(2+1)D 18](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.video.r2plus1d_18.html), [MC3 18](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.video.mc3_18.html), [Video MViT](https://docs.pytorch.org/vision/main/models/video_mvit.html)

**Dos Estrategias para Usar Estos Modelos**

Existen dos estrategias principales para integrar estos modelos en tu trabajo:

* **Inferencia (Predicción Directa)**: Utilizas el conocimiento existente de un modelo directamente para realizar predicciones. Esto es perfecto cuando tu tarea es muy similar a aquello para lo que el modelo fue entrenado originalmente.

* **Transfer Learning (Ajuste Fino o Fine-Tuning)**: Adaptas un modelo preentrenado para una tarea nueva y especializada aprovechando su conocimiento experto previo. Esta es una estrategia altamente flexible que te brinda un control preciso sobre cómo el modelo aprende tus nuevos datos. El enfoque puede variar desde entrenar solo una nueva capa final sobre el modelo base congelado hasta ajustar selectivamente capas más profundas, o incluso reentrenar toda la red en tu dataset personalizado. Esta técnica tan potente te permite lograr resultados excelentes mucho más rápido y con menos datos que entrenando desde cero.


En este laboratorio, te enfocarás completamente en la primera estrategia: **realizar inferencia**. Verás el transfer learning en acción en el próximo laboratorio.

### Conocer lo que sabe tu modelo: Clases y capacidades

Antes de usar un modelo preentrenado, debes realizar una comprobación de coherencia rápida pero vital. Piensa en esto como leer la etiqueta de una herramienta antes de usarla: necesitas saber exactamente para qué fue diseñada. Específicamente, necesitas responder a dos preguntas:

* **¿Cuántas clases** puede predecir este modelo?
* **¿Cuáles son los nombres** de esas clases?

Esta sencilla investigación evita grandes dolores de cabeza. Por ejemplo, si tu proyecto requiere detectar perros, primero debes confirmar que el modelo que has elegido incluya "perro" en su lista de clases reconocidas. Responder a esta pregunta de antemano te evita perder tiempo en un modelo que no puede realizar tu tarea específica.


La forma de encontrar esta información depende de la antigüedad del modelo. Aquí tienes un enfoque genérico de dos pasos que puedes aplicar a cualquier modelo.

#### Método 1: El enfoque moderno (verificar primero los metadatos)

Los modelos modernos de TorchVision facilitan esto al incrustar esta información directamente en el objeto de pesos (`weights`) del modelo bajo un atributo `.meta`. Esta es la fuente de verdad oficial y más confiable, y **siempre deberías verificar esto primero**.

Para agilizar este proceso, utilizarás la función auxiliar proporcionada `get_model_classes_from_weights_meta`, que inspecciona automáticamente el objeto de pesos e imprime la lista de clases por ti.

In [ ]:
def get_model_classes_from_weights_meta(model, weights_obj=None):
    """
    Inspecciona el objeto de pesos de un modelo para encontrar y devolver los nombres de las clases.
    
    Args:
        model: La instancia del modelo.
        weights_obj: El objeto de enumeración de pesos (ej. ResNet50_Weights.DEFAULT).
    
    Returns:
        Una tupla que contiene (numero_de_clases, lista_de_nombres_de_clases), o (None, None) si no se encuentran.
    """
    num_classes = None
    class_names = None

    # Comprueba si se proporcionó un objeto de pesos y si tiene los metadatos necesarios
    if weights_obj and hasattr(weights_obj, 'meta') and "categories" in weights_obj.meta:
        class_names = weights_obj.meta["categories"]
        num_classes = len(class_names)
        print(f"El modelo está configurado para {num_classes} clases según los metadatos de los pesos (Weights Metadata). Estas clases son:\n")
        
        # Para una impresión estética, mostramos la lista
        pprint(class_names)       
        return num_classes, class_names
    else:
        print("No se encontraron metadatos de 'categories' para este modelo.")
        return num_classes, class_names

* Para este ejemplo, inspeccionarás un modelo [deeplabv3_resnet50](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.segmentation.deeplabv3_resnet50.html), que se utiliza para la segmentación de imágenes.
* De los [pesos de este modelo en particular](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.segmentation.deeplabv3_resnet50.html#torchvision.models.segmentation.DeepLabV3_ResNet50_Weights), utilizarás `DeepLabV3_ResNet50_Weights.DEFAULT`. Aquí, `.DEFAULT` es un alias que apunta automáticamente a los mejores y más actuales pesos preentrenados disponibles para este modelo.
    * El uso de `.DEFAULT` se considera una mejor práctica, ya que hace que tu código sea más robusto y esté preparado para el futuro.

In [ ]:
# Definir el modelo en sí mismo sin pesos por ahora
seg_model = tv_models.segmentation.deeplabv3_resnet50(weights=None)

# Seleccionar los pesos preentrenados específicos que deseas inspeccionar para el modelo seleccionado
seg_model_weights = tv_models.segmentation.DeepLabV3_ResNet50_Weights.DEFAULT

* Use the helper function to get the class information.

In [ ]:
# Call the helper function to inspect the provided weights object.
num_classes, class_names_deeplabv3 = get_model_classes_from_weights_meta(
    model=seg_model,              # The model architecture.
    weights_obj=seg_model_weights # The weights object containing the .meta attribute to inspect.
)

<br>

Como puedes ver, la función auxiliar extrajo con éxito los 21 nombres de las clases directamente del objeto de pesos. Ahora puedes decidir con confianza si este modelo, con estos pesos específicos, es el adecuado para tu tarea.

Retomando el ejemplo anterior, si tu objetivo fuera encontrar un "perro", ahora puedes escanear esta lista y confirmar que `'dog'` es una clase reconocida. Esto te indica que el modelo es una elección adecuada. Si "dog" no hubiera estado en la lista, sabrías de inmediato que necesitas buscar un modelo y unos pesos diferentes, ahorrándote un tiempo valioso.

#### Método 2: El enfoque manual (cuando no existen metadatos)

¿Qué haces cuando un modelo no tiene el atributo `.meta`? Este es un escenario común que encontrarás con modelos más antiguos cargados a través de la bandera `pretrained=True`, o con modelos personalizados que no tienen esta información empaquetada con ellos.

En estas situaciones, necesitas ponerte el sombrero de detective. Investigarás la arquitectura del modelo directamente para descubrir la información que necesitas. Esta inspección manual es una habilidad fundamental, que garantiza que puedas trabajar de manera efectiva con cualquier modelo, no solo con los más nuevos.

* Usemos un modelo [Resnet50](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.resnet50.html) más antiguo como ejemplo.
    * Lo cargarás usando el método heredado (legacy) `pretrained=True`, que obtiene automáticamente sus pesos estándar preentrenados en el dataset ImageNet 1K.

Primero, confirma que este método de carga, de hecho, carece de un atributo `.meta` detectable.

In [ ]:
# Instantiate the ResNet50 model using the legacy `pretrained=True` method.
resnet50_model = tv_models.resnet50(pretrained=True)

Cuando usas `pretrained=True`, los pesos se cargan directamente en las capas del modelo. A diferencia del enfoque moderno, esto no crea un objeto de pesos independiente que contenga los `metadata`. Por lo tanto, pasarás explícitamente `weights_obj=None` a la función auxiliar para demostrar que no existe tal objeto disponible con este método.

In [ ]:
# Se pasa `weights_obj=None` porque el método de carga `pretrained=True`
# no crea un objeto de pesos independiente que tenga un atributo .meta para inspeccionar.
num_classes, class_names = get_model_classes_from_weights_meta(
    model=resnet50_model, 
    weights_obj=None
)   

<br>

Esto confirma que este método de carga no proporciona metadatos, razón por la cual debes proceder con la inspección manual de la arquitectura del modelo.

#### Inspección Manual de la Arquitectura

Dado que el enfoque de los metadatos fue un callejón sin salida, tu siguiente paso es inspeccionar el plano del modelo directamente. Tu objetivo es encontrar la última capa de la red, ya que su configuración te dirá cuántas clases predice el modelo.

La pista específica que estás buscando es el parámetro `out_features` en la capa final `Linear` (completamente conectada) del modelo. La forma más sencilla de encontrar esto es imprimir el objeto del modelo en sí:

```
print(resnet50_model)
```

Esto imprimirá toda la arquitectura del modelo, que puede ser bastante larga. Solo necesitas concentrarte en las últimas líneas de la salida. Para `ResNet50`, encontrarás una capa llamada `fc` que contiene la respuesta:

```
...
  (avgpool): AdaptiveAvgPool2d(output_size=(1, 1))
  (fc): Linear(in_features=2048, out_features=1000, bias=True)
)
```

In [ ]:
# ### Comenta y ejecuta la línea de abajo si deseas imprimir la arquitectura del modelo.
# print(resnet50_model)

<br>

El valor `out_features=1000` te indica que el modelo está configurado para 1000 clases. Puedes acceder a este valor directamente en el código para confirmarlo.

In [ ]:
# Obtener el número de características de salida de la capa llamada 'fc'
num_classes = resnet50_model.fc.out_features

print(f"Inspeccionando la capa .fc del modelo: Tiene {num_classes} clases de salida.")

#### Encontrar los nombres de las clases

Tu trabajo de detective está casi completo. Has descubierto que el modelo predice 1000 clases, pero un **índice** de clase (como `248`) no tiene sentido sin un **nombre** de clase (como `'bull mastiff'`). El paso final es encontrar la leyenda que mapea estos índices con nombres legibles por humanos. Como has aprendido, esta información se almacena de forma **externa** al archivo del modelo.

Para un modelo como `ResNet50` (variante heredada) entrenado en `ImageNet`, esta lista de clases está bien documentada. Estos son los primeros lugares donde un profesional buscaría:

* **Documentación oficial: Comienza siempre por aquí**. La documentación de PyTorch para un modelo o sus pesos casi siempre describirá el dataset en el que fue entrenado y proporcionará un enlace a la lista de clases.

* **Archivos auxiliares de la comunidad**: Los repositorios de código y los tutoriales en línea son otra excelente fuente. A menudo puedes encontrar archivos auxiliares (como un `.json` o `.txt`) que contienen el mapeo directo de índice a nombre.

En la siguiente sección, utilizarás uno de estos archivos auxiliares para cargar los nombres de las clases de ImageNet desde un archivo `.json` y luego realizar la clasificación.

## Realización de Inferencia con Modelos Preentrenados

Ahora que sabes cómo investigar las capacidades de un modelo, es hora de poner ese conocimiento en práctica. Realizarás una **inferencia** utilizando un modelo "listo para usar" para obtener predicciones sobre imágenes nuevas.

Aplicarás este principio a los dos modelos que ya has inspeccionado:

* **Segmentación de Imágenes**: Usarás `DeepLabV3` para encontrar y dibujar una máscara perfecta a nivel de píxel sobre un objeto.

* **Clasificación de Imágenes**: Usarás la versión heredada (legacy) de `ResNet50` para predecir el sujeto principal de una imagen.

### Segmentación de Imágenes

Tu primera tarea es realizar la **segmentación de imágenes** con el modelo `DeepLabV3`. El objetivo es sencillo: encontrar un perro en una imagen y generar una máscara perfecta a nivel de píxel que resalte su forma exacta.

* Comenzarás cargando una imagen.

In [ ]:
# Define the file path for the image.
image_path = './images/dog2.jpg'

# Display the image.
DisplayImage(image_path, width=300, height=450)

<br>

En tu investigación anterior, confirmaste que la lista de clases del modelo `DeepLabV3` incluye `'dog'`, lo que lo convierte en la herramienta adecuada para esta tarea.

* Ahora cargarás el modelo nuevamente, esta vez adjuntando sus pesos preentrenados `.DEFAULT` y llamando a `.eval()` para prepararlo para la inferencia.
    * **Nota sobre la descarga**: Ahora se descargará un archivo más grande porque estás cargando el modelo completo (backbone + head). La lista de clases que verificaste anteriormente sigue siendo correcta, ya que todavía estás utilizando los pesos `.DEFAULT`.

In [ ]:
# Instancia el modelo de la arquitectura y carga los pesos preentrenados.
seg_model = tv_models.segmentation.deeplabv3_resnet50(weights=seg_model_weights).eval()

Con tu `seg_model` preentrenado cargado y en modo de evaluación, ahora recorrerás el proceso completo para realizar la segmentación de imágenes.

* **Prepara tus tensores**: Comienza cargando la imagen. A partir de ella, crearás dos versiones de tensores:

    * `original_image_tensor`: Un tensor limpio y sin normalizar que usarás más adelante para la visualización.
    
    * `input_tensor`: Una versión normalizada de ese tensor que se utilizará como entrada para el modelo.
    
* **Define tus objetivos y colores**: A continuación, definirás una lista `target_class_names`, estableciéndola solo como `['dog']` para este ejemplo. También crearás una lista `seg_colors` correspondiente para asignar un color a la máscara de cada objetivo. Finalmente, obtendrás los índices de clase de la lista `class_names_deeplabv3`, la cual recuperaste en la sección del "**Enfoque Moderno**".

    * **NOTA:** Cada nombre de clase en tu lista `target_class_names` debe ser una **coincidencia exacta y sensible a mayúsculas** con un nombre en la lista de clases del modelo; de lo contrario, la búsqueda fallará.

In [ ]:
# Cargar la imagen PIL base
img = Image.open(image_path)

# Crear el tensor limpio y sin normalizar para la visualización posterior
original_image_tensor = transforms.ToTensor()(img)

# Definir la transformación de normalización
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])

# Crear el tensor de entrada normalizado para el modelo
# .unsqueeze(0) añade una dimensión de lote (batch), cambiando la forma del tensor
# de [C, H, W] a [N, C, H, W], ya que los modelos esperan un lote de imágenes como entrada
input_tensor = normalize(original_image_tensor).unsqueeze(0)

# Definir una lista de clases objetivo que deseas encontrar
target_class_names = ['dog'] # Se pueden añadir más clases también, ej., ['dog', 'person', ...]

# Definir una lista de colores correspondiente para las máscaras de segmentación de cada clase
seg_colors = ["blue"]

# Usar una comprensión de lista para obtener una lista de los índices de clase correspondientes
class_indices = [class_names_deeplabv3.index(name) for name in target_class_names]

# Imprimir los resultados para confirmación
print(f"Clases Objetivo:       {target_class_names}")
print(f"Índices Correspondientes: {class_indices}")

<br>

* **Realizar la Inferencia**: Pasa el `input_tensor` normalizado al `seg_model` preentrenado. El modelo devuelve sus predicciones en un diccionario, por lo que usarás `['out'][0]` para acceder a las puntuaciones de salida brutas (logits) de la primera (y única) imagen del lote.

In [ ]:
# Generate prediction
with torch.no_grad():
    output = seg_model(input_tensor)['out'][0]

* **Generar la Máscara Final**: Convierte las puntuaciones brutas en una **pila de máscaras booleanas**, una para cada una de tus clases objetivo (en este caso, solo para la clase `'dog'`). Esto implica tres pasos:

    * Primero, usar `.argmax(0)` para obtener la mejor predicción individual del modelo para cada píxel.

    * Segundo, crear una lista de **máscaras booleanas individuales**, donde cada máscara corresponde a una de tus clases objetivo.

    * Finalmente, usar `torch.stack()` para combinar esta lista de máscaras en un solo tensor. Este es el formato requerido para dibujar múltiples máscaras con diferentes colores.

In [ ]:
# Obtener la clase predicha para cada píxel encontrando la clase con la puntuación más alta.
output_predictions = output.argmax(0)

# Crear una máscara booleana independiente para cada una de tus clases objetivo.
# El resultado es una lista de tensores booleanos, uno para cada índice de clase.
individual_masks = [(output_predictions == i) for i in class_indices]

# Apilar (stack) las máscaras individuales en un único tensor de forma (num_masks, H, W).
stacked_masks = torch.stack(individual_masks, dim=0)

* **Dibujar Máscara**: Usa `draw_segmentation_masks` para superponer tus máscaras generadas sobre la imagen original, pasando estos argumentos clave:

    * `image=(original_image_tensor * 255).byte()`: Convierte el `original_image_tensor` de un `float` (escala 0.0-1.0) a un tensor `uint8` (escala 0-255), según lo requiere la función.
    
    * `masks=stacked_masks`: El tensor de forma `(num_masks, H, W)` que creaste, donde cada capa es una máscara booleana para una de tus clases objetivo.

    * `colors=seg_colors`: La lista de cadenas de colores que definiste. La función los usará para dibujar cada máscara correspondiente en el tensor `stacked_masks`.

In [ ]:
# Aplicar las máscaras de segmentación usando el tensor stacked_masks.
result = vutils.draw_segmentation_masks(image=(original_image_tensor * 255).byte(),
                                        masks=stacked_masks,
                                        alpha=0.5,
                                        colors=seg_colors)
# Visualizar la máscara
helper_utils.display_images(processed_image=result, figsize=(7, 7))

### Clasificación de Imágenes

Para tu segunda tarea de inferencia, cambiarás a la **clasificación de imágenes**. Usando la misma imagen que antes, tu objetivo es que un modelo prediga la clase única más probable para toda la imagen. Para esto, utilizarás el modelo [Resnet50](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.resnet50.html).

* Carga el modelo con sus pesos heredados (legacy) estableciendo `pretrained=True`.
    * También establece el modelo en modo de evaluación.

In [ ]:
# Cargar el modelo ResNet50, usando los pesos heredados y configurarlo a .eval()
resnet50_model = tv_models.resnet50(pretrained=True).eval()

Tu investigación manual anterior reveló que el modelo `ResNet50` **con estos pesos heredados (`pretrained=True`)** predice 1000 clases. Para dar sentido a estas predicciones, necesitas encontrar la lista correspondiente de nombres de clases.

Como has aprendido, esta información no viene empaquetada con el modelo en sí. Para un dataset estándar como ImageNet, esta "leyenda" se distribuye comúnmente en un archivo `.json` externo. En la siguiente celda:

* Usarás la función auxiliar `load_imagenet_classes` para cargar los mapeos de clase desde el archivo `'./imagenet_class_index.json'`.

* Esto crea un **diccionario** de Python que mapea cada índice de clase con su nombre legible por humanos.

**Nota**: Este paso manual de cargar un archivo externo solo es necesario porque utilizaste el método heredado `pretrained=True`. Si hubieras utilizado un objeto de pesos moderno (por ejemplo, `ResNet50_Weights.IMAGENET1K_V2`), podrías haber usado simplemente el atributo `.meta`, tal como hiciste con `DeepLabV3`.

In [ ]:
# Use la función auxiliar para cargar las asignaciones de índice a nombre de clase desde el archivo JSON.
imagenet_classes = helper_utils.load_imagenet_classes('./imagenet_class_index.json')

<br>

* En lugar de imprimir las 1000 clases, la celda de código a continuación está configurada para inspeccionar una "porción" específica del diccionario, comenzando en el índice `200`. Este rango contiene varias razas de perros.

* Cuando ejecutes la celda, presta atención al **índice `207`**. Verás que corresponde a `'golden_retriever'`, la coincidencia perfecta para el perro de la imagen que estás a punto de clasificar.

    * Después de ejecutarlo una vez, siéntete libre de cambiar los valores de `start_index` y `num_to_print` para explorar otras secciones de la lista de clases.

In [ ]:
print("Total de Clases:", len(imagenet_classes), "\n")

# Definir el índice de inicio y cuántas clases deseas imprimir
start_index = 200
num_to_print = 10

print(f"Imprimiendo {num_to_print} clases empezando desde el índice {start_index}:\n")

# Bucle a través del rango deseado de índices
for i in range(start_index, start_index + num_to_print):
    key = str(i)
    value = imagenet_classes[key]
    print(f"Índice {key}: {value}")

<br>

Con tu modelo `ResNet50` y los nombres de las clases cargados, ahora recorrerás el proceso de clasificación de la imagen.

* **Preparar la Imagen**: Define un pipeline de `transform` para redimensionar, recortar y normalizar la imagen para que coincida con el formato en el que se entrenó el modelo. Luego, aplicarás este pipeline a tu imagen y usarás `.unsqueeze(0)` para crear un lote de uno para la entrada del modelo.

**Nota Importante**: Al preparar una imagen para un modelo preentrenado, es **muy recomendable** que el tensor de entrada final tenga las **mismas dimensiones** y **normalización** que los datos con los que se entrenó el modelo. Modelos como `ResNet50` esperan un tamaño de entrada específico (por ejemplo, 224x224 píxeles) y una distribución de datos determinada. Una discrepancia en cualquiera de estos aspectos puede causar errores o degradar significativamente el rendimiento. El pipeline de transformación a continuación es el método estándar para lograr este formato para modelos entrenados en ImageNet.

In [ ]:
# Cargar la imagen PIL base
img = Image.open(image_path)

# Definir el pipeline de transformación
transform = transforms.Compose([
    transforms.Resize(256),
    # Modelos como ResNet50 esperan una entrada de 224x224, por lo que redimensionas a una imagen
    # ligeramente más grande y luego tomas un recorte central de las dimensiones objetivo.
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    # Normalizar el tensor con la media y la desviación estándar del dataset ImageNet.
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Aplicar las transformaciones y añadir una dimensión de lote (batch)
input_tensor = transform(img)
# La capa de entrada del modelo espera un tensor 4D con una forma específica: [N, C, H, W]
input_batch = input_tensor.unsqueeze(0)

* **Realizar la Inferencia**: Pasa el `input_batch` preparado al `resnet50_model` preentrenado. El modelo producirá un tensor de puntuaciones brutas (logits) sin normalizar, una puntuación por cada una de las 1000 clases de ImageNet.

In [ ]:
# Perform Inference
with torch.no_grad():
    output = resnet50_model(input_batch)

* **Convertir Puntuaciones en Probabilidades**: Aplica la función <code>[torch.nn.functional.softmax()](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html)</code> a los logits.
    * Esto convierte las puntuaciones brutas en una distribución de probabilidad donde cada valor representa la confianza del modelo para una clase en particular, y todas las probabilidades suman 1.

In [ ]:
# Aplicar softmax para obtener probabilidades
# Aplicarlo al primer (y único) elemento en el lote de salida.
probabilities = torch.nn.functional.softmax(output[0], dim=0)

* **Get the Top Predictions**: Use <code>[torch.topk()](https://docs.pytorch.org/docs/stable/generated/torch.topk.html)</code> to efficiently find the `top` classes with the highest probabilities and get their corresponding class IDs (indices).

In [ ]:
# Obtener las `top` probabilidades y sus IDs de clase correspondientes
top = 5
top_prob, top_catid = torch.topk(probabilities, top)

* **Mostrar los Resultados**: Realiza un bucle a través de los resultados `top`. Para cada uno, usarás su ID de clase para buscar el nombre legible por humanos en el diccionario `imagenet_classes` e imprimirlo junto con su puntuación de confianza.

In [ ]:
# Convert IDs to class names and print results
print(f"Top {top} predictions:")
for i in range(top_prob.size(0)):
    # Get the string representation of the class ID
    class_id_str = str(top_catid[i].item())
    
    # Look up the class name in the dictionary
    class_name = imagenet_classes[class_id_str][1]
    confidence = top_prob[i].item() * 100
    print(f"\tTop-{i+1}: {class_name} ({confidence:.2f}%)")

<br>

Como puedes ver, el modelo identificó correctamente al perro como un **`'golden_retriever'`** con una alta confianza, concluyendo con éxito el flujo de trabajo principal del laboratorio. Este laboratorio te ha guiado a través de las habilidades esenciales de un profesional de la visión por computadora: investigar las capacidades de un modelo, utilizar modelos preentrenados para la inferencia y emplear las utilidades de `torchvision` para visualizar los resultados.

## (Opcional) Detección de Objetos

Con esta base sólida, la sección final ofrece una oportunidad práctica para ver estos conceptos en acción en tus propias imágenes. Ejecutarás el código proporcionado que utiliza un potente modelo `Faster R-CNN` para realizar la **detección de objetos** y **DeepLabV3** para generar máscaras de segmentación. Esta es una oportunidad para experimentar con las herramientas proporcionadas y observar sus efectos en datos nuevos.

### Detección de objetos mediante el modelo `fasterrcnn_resnet50_fpn`

El modelo [fasterrcnn_resnet50_fpn](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn.html) es una opción popular y potente para tareas de detección de objetos gracias a las fortalezas de sus componentes combinados:

* `Faster R-CNN (Region-based Convolutional Neural Network)`: Esta es la arquitectura principal diseñada específicamente para la detección de objetos. Identifica de manera eficiente las regiones potenciales de objetos (Region Proposal Network) y luego clasifica estas regiones y perfecciona las coordenadas de sus cajas delimitadoras (bounding boxes). Es conocida por su buen equilibrio entre velocidad y precisión.


* `ResNet-50 (Residual Network 50)`: Sirve como el "backbone" (columna vertebral) del modelo. ResNet-50 es una red neuronal convolucional profunda de 50 capas, famosa por su capacidad para aprender características ricas y jerárquicas de las imágenes. Sus "conexiones residuales" ayudan a entrenar redes muy profundas de manera efectiva, lo que conduce a un mejor rendimiento.

* `FPN (Feature Pyramid Network)`: Este componente mejora la capacidad del modelo para detectar objetos a múltiples escalas. La FPN construye una pirámide de mapas de características con diferentes resoluciones, lo que permite al modelo identificar eficazmente objetos tanto pequeños como grandes dentro de la misma imagen.


En esencia, `fasterrcnn_resnet50_fpn` aprovecha un potente extractor de características (ResNet-50), una estrategia de detección multiescala eficaz (FPN) y un marco de detección de objetos robusto (Faster R-CNN), lo que lo convierte en un modelo equilibrado y de alto rendimiento para localizar y clasificar diversos objetos en una imagen. El uso de `FasterRCNN_ResNet50_FPN_Weights.DEFAULT` garantiza que estás cargando el modelo con pesos preentrenados, típicamente entrenados en un dataset grande como COCO, lo que le permite reconocer una amplia variedad de objetos comunes de inmediato.

* Ejecuta la siguiente celda para cargar el modelo y establecerlo en modo de evaluación.

In [ ]:
# Load a pre-trained object detection model and set to evaluation mode
bb_model_weights = tv_models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
bb_model = tv_models.detection.fasterrcnn_resnet50_fpn(weights=bb_model_weights).eval()

#### ¿Cuántas clases puede detectar el modelo?

In [ ]:
# Use the helper function to inspect the weights object of the object detection model.
num_classes, classes = get_model_classes_from_weights_meta(
    model=bb_model, 
    weights_obj=bb_model_weights
)

* Como habrás **notado**, algunos índices de clase tienen un `N/A` al lado.
* Las `91` clases de salida del modelo incluyen `81` clases reconocibles del dataset COCO, más 10 entradas adicionales marcadas como `N/A` (cuya naturaleza específica no se define en esta lista).
* Si utilizaras una lista estándar de 81 clases de COCO en lugar de esta lista específica del modelo, encontrarías discrepancias de índices con la salida del modelo.



* Tu objetivo para este ejemplo es detectar coches y semáforos. Primero, establecerás la ruta a una imagen que contenga varios de ellos.

In [ ]:
# Set the file path to the image you'll use for car/traffic light detection.
image_path = './images/cars.jpg'

In [ ]:
# Display the image
DisplayImage(image_path, width=600, height=550)

* Define `target_class_names` como una lista de cadenas que especifiquen los objetos que deseas detectar, en este caso, `['car', 'traffic light']`.

* Crea una lista `bbox_colors` con una cadena de color correspondiente para la caja delimitadora (bounding box) de cada clase objetivo.

* Genera la lista `object_indices` a partir de los nombres de los objetivos utilizando una comprensión de lista.

In [ ]:
# Define a list of target classes to detect
target_class_names = ['car', 'traffic light']

# Define a corresponding list of colors for each class's bounding box
bbox_colors = ['red', 'blue']

# Use a list comprehension to get a list of all target indices
object_indices = [classes.index(name) for name in target_class_names]

Se te proporciona la función `detect_and_draw_bboxes`. Esta toma el modelo de detección, la ruta de la imagen, las listas de objetos objetivo y sus colores, y un umbral de confianza. Luego, devuelve la imagen con las cajas delimitadoras dibujadas para todos los objetos objetivo detectados.

La funcionalidad principal de esta función es la siguiente:

* **Uso de un umbral (`threshold`) de confianza**: Se utiliza una puntuación de confianza mínima para filtrar las detecciones dudosas.

* **Predicción y puntuación del modelo**: El modelo procesa la imagen de entrada, genera posibles cajas delimitadoras para todos los objetos detectados y asigna puntuaciones de confianza a cada una.

* **Filtrado de cajas calificadas**: La función recorre tu lista de objetivos y selecciona solo aquellas cajas que:

    * Coincidan con una de las **clases de objetos objetivo** que proporcionaste.
    
    * Tengan una puntuación de confianza mayor que el `threshold` establecido.

* **Dibujado y retorno**: La función recopila todas las cajas calificadas de todas tus clases objetivo y luego utiliza la utilidad `draw_bounding_boxes` para dibujarlas en la imagen, cada una con su etiqueta y color especificados. Se devuelve la imagen final modificada (o la original, si ninguna caja calificó).

In [ ]:
def detect_and_draw_bboxes(model, image_path, object_indices, labels, bbox_colors, threshold, bbox_width=3):
    """
    Detecta y dibuja cajas delimitadoras (bounding boxes) etiquetadas para múltiples clases de objetos especificadas en una imagen.

    Args:
        model: Modelo de detección de objetos preentrenado.
        image_path (str): Ruta al archivo de imagen.
        object_indices (list): Lista de índices para las clases objetivo a detectar.
        labels (list): Lista de etiquetas de texto para cada clase objetivo.
        bbox_colors (list): Lista de colores para las cajas delimitadoras de cada clase objetivo.
        threshold (float): Umbral de confianza para las detecciones.
        bbox_width (int, opcional): El ancho de línea para las cajas delimitadoras. Por defecto es 3.

    Returns:
        torch.Tensor: Tensor de la imagen con todas las cajas detectadas dibujadas.
    """
    
    # Abrir y transformar la imagen, y preparar el tensor de resultado
    pil_image = Image.open(image_path).convert("RGB")
    transform_to_tensor = transforms.Compose([transforms.ToTensor()])
    tensor_image_batch = transform_to_tensor(pil_image).unsqueeze(0)
    result_image_tensor = (tensor_image_batch.squeeze(0) * 255).byte()

    # Realizar la inferencia para obtener predicciones de todos los objetos posibles
    with torch.no_grad():
        prediction = model(tensor_image_batch)[0]

    # Inicializar listas para recolectar todas las cajas, etiquetas y colores que cumplan con los criterios
    all_boxes_to_draw = []
    all_labels_to_draw = []
    all_colors_to_draw = []

    # Recorrer cada clase objetivo para encontrar sus cajas
    for index, label, color in zip(object_indices, labels, bbox_colors):
        # Filtrar predicciones para el índice de clase actual y el umbral de confianza
        class_mask = (prediction['labels'] == index) & (prediction['scores'] > threshold)
        
        # Obtener las cajas para la clase actual
        boxes_for_this_class = prediction['boxes'][class_mask]

        if boxes_for_this_class.nelement() > 0:
            # Añadir las cajas encontradas a nuestra lista maestra
            all_boxes_to_draw.extend(boxes_for_this_class.tolist())
            # Crear y añadir las etiquetas y colores correspondientes
            all_labels_to_draw.extend([label] * len(boxes_for_this_class))
            all_colors_to_draw.extend([color] * len(boxes_for_this_class))

    # Después de verificar todas las clases, dibujar todas las cajas recolectadas a la vez si se encontró alguna
    if all_boxes_to_draw:
        result_image_tensor = vutils.draw_bounding_boxes(
            result_image_tensor,
            torch.tensor(all_boxes_to_draw),
            labels=all_labels_to_draw,
            colors=all_colors_to_draw,
            width=bbox_width
        )
    else:
        # Si la lista de cajas a dibujar está vacía, imprimir esta información.
        print(f"No se encontraron objetos de la lista {labels} con una puntuación de confianza superior a {threshold}.\n")
    
    return result_image_tensor

* Establece un umbral (threshold) de confianza. Este valor (de 0.0 a 1.0) es la puntuación mínima que debe tener una detección para ser considerada válida.
    * Siéntete libre de experimentar con este valor. Un umbral más bajo (ej. `0.3`) podría revelar más objetos, mientras que uno más alto (ej. `0.8`) solo mostrará las detecciones con mayor confianza.

In [ ]:
confidence_threshold = 0.7


* Ejecuta la función para detectar cajas delimitadoras de `car` (coche) y `traffic light` (semáforo) en la imagen.

In [ ]:
# Execute the main detection function
result_image_tensor = detect_and_draw_bboxes(
    model=bb_model,                  # The pre-trained object detection model.
    image_path=image_path,           # The path to the input image.
    object_indices=object_indices,   # The list of integer indices for the target classes.
    labels=target_class_names,       # The list of string names for the box labels.
    bbox_colors=bbox_colors,         # The list of colors for the bounding boxes.
    threshold=confidence_threshold,  # The minimum confidence score for a detection.
)

# Display the results
helper_utils.display_images(processed_image=result_image_tensor, figsize=(15, 15))

### ¡Pruébalo tú mismo!

Aquí es donde verás cómo se une todo lo que has aprendido. En esta sección, utilizarás los potentes modelos y funciones de este laboratorio para realizar la **detección de objetos** y la **segmentación de imágenes** en tus propias fotos. Podrás subir una imagen, seleccionar tus objetos objetivo cambiando las variables de entrada y luego ejecutar los modelos preentrenados para verlos en acción con tus propios datos.


### 1 - Encuentra cajas delimitadoras en objetos de tus propias imágenes

Al ejecutar la función `helper_utils.upload_jpg_widget()` se mostrará un widget que te permitirá subir tus propias imágenes al espacio de trabajo.

* Solo puedes subir imágenes que tengan una extensión `.jpg`.
* Cada imagen no debe exceder los **5 MB** de **tamaño de archivo**.
* Una vez que la imagen se haya subido correctamente, verás su ruta de archivo en pantalla, la cual puedes copiar y pegar directamente en la celda de abajo.

Además, una vez que se muestre el widget, puedes usarlo varias veces para subir imágenes; no tienes que volver a ejecutar la función `helper_utils.upload_jpg_widget()`.

In [ ]:
helper_utils.upload_jpg_widget()

* Establece la ruta de tu imagen (tal como se muestra arriba).

* Alternativamente, puedes usar estas imágenes que ya están presentes en el espacio de trabajo:
  ```
  image_path = './images/birds_sheep_dog.jpg'
  image_path = './images/car_bus_tram.jpg'
  image_path = './images/person_and_bicycle.jpg'
  ```
<br>  
* Se ha establecido una ruta por defecto para ti, pero siéntete libre de cambiarla por una diferente.

In [ ]:
image_path = './images/car_bus_tram.jpg' ### Add your image path here 

In [ ]:
# Display the image
DisplayImage(image_path, width=500, height=500)

* Como recordatorio, a continuación se muestran las clases que el modelo puede detectar.


In [ ]:
detection_classes = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A',
    'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
    'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A',
    'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

* En la celda de código a continuación, definirás los parámetros para tu tarea de detección de objetos:

    * `target_class_names`: Una lista de cadenas para todos los objetos que deseas detectar. Cada nombre debe coincidir exactamente con la lista `detection_classes` anterior.

    * `bbox_colors`: Una lista de cadenas de colores correspondiente para la caja delimitadora de cada clase objetivo.

    * `confidence_threshold`: Un valor entre 0.0 y 1.0 que establece la confianza mínima para que se muestre una detección.

* Se han establecido valores de ejemplo para ti. Siéntete libre de cambiar estos valores para detectar diferentes objetos en tu imagen o para ajustar la sensibilidad de la detección.

In [ ]:
# Define a list of target classes to detect
target_class_names = ['car', 'bus', 'train'] ### Add your target class names here

# Define a corresponding list of colors for each class's bounding box
bbox_colors = ['red', 'purple', 'green'] ### Add your target class names here

# Set the confidence_threshold
confidence_threshold = 0.7 ### Set your threshold here

* Check out the results!

In [ ]:
# Usar una comprensión de lista para obtener una lista de todos los índices objetivo
object_indices = [detection_classes.index(name) for name in target_class_names]

# Definir la etiqueta para el objeto igual que target_class_names
labels = target_class_names

# Ejecutar la función principal de detección
result_image_tensor = detect_and_draw_bboxes(
    model=bb_model,                  # El modelo de detección de objetos preentrenado.
    image_path=image_path,           # La ruta a la imagen de entrada.
    object_indices=object_indices,   # La lista de índices enteros para las clases objetivo.
    labels=labels,                   # La lista de nombres de cadena para las etiquetas de las cajas.
    bbox_colors=bbox_colors,         # La lista de colores para las cajas delimitadoras.
    threshold=confidence_threshold,  # La puntuación de confianza mínima para una detección.
    bbox_width=5                     # El grosor de línea para las cajas delimitadoras.
)

# Mostrar los resultados (siéntete libre de establecer un `figsize` diferente)
helper_utils.display_images(processed_image=result_image_tensor, figsize=(10, 10))

### 2 - Generación de segmentación en objetos de tus propias imágenes

Ejecuta la función `helper_utils.upload_jpg_widget()` para mostrar el widget de carga de imágenes.

In [ ]:
helper_utils.upload_jpg_widget()

* Establece la ruta de tu imagen (tal como se muestra arriba).

* Alternativamente, puedes usar estas imágenes que ya están presentes en el espacio de trabajo:
  ```
  image_path = './images/birds_sheep_dog.jpg'
  image_path = './images/car_bus_tram.jpg'
  image_path = './images/person_and_bicycle.jpg'
  ```
<br>  
* Se ha establecido una ruta por defecto para ti, pero siéntete libre de cambiarla por una diferente.

In [ ]:
image_path = './images/person_and_bicycle.jpg' ### Add your image path here

In [ ]:
# Display the image
DisplayImage(image_path, width=500, height=500)

* Como recordatorio, a continuación se muestran las clases que el modelo de segmentación puede detectar.

In [ ]:
segmentation_classes = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car',
    'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

En la celda de código a continuación, definirás los parámetros para tu tarea de segmentación de imágenes:

* `target_class_names`: Una lista de cadenas para todos los objetos que deseas segmentar. Cada nombre debe coincidir exactamente con la lista `segmentation_classes` anterior.

* `seg_colors`: Una lista de cadenas de colores correspondiente para la máscara de segmentación de cada clase objetivo.

* Se han establecido valores de ejemplo para ti. Siéntete libre de cambiar estos valores para segmentar diferentes objetos en tu imagen o para usar colores diferentes para las máscaras.

In [ ]:
# Define una lista de clases objetivo que deseas encontrar
target_class_names = ['person', 'bicycle'] ### Añade aquí el nombre de tus clases objetivo

# Define una lista correspondiente de colores para las máscaras de segmentación de cada clase
seg_colors = ["pink", 'yellow'] ### Añade aquí los colores de las máscaras para cada clase

* Check out the results!

In [ ]:
# Cargar el modelo de segmentación preentrenado
seg_model_weights = tv_models.segmentation.DeepLabV3_ResNet50_Weights.DEFAULT
seg_model = tv_models.segmentation.deeplabv3_resnet50(weights=seg_model_weights).eval()

# Cargar la imagen PIL base
img = Image.open(image_path)

# Crear el tensor limpio y sin normalizar para la visualización posterior
original_image_tensor = transforms.ToTensor()(img)

# Definir la transformación de normalización
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])

# Crear el tensor de entrada normalizado para el modelo
input_tensor = normalize(original_image_tensor).unsqueeze(0)

# Usar una comprensión de lista para obtener una lista de los índices de clase correspondientes
class_indices = [segmentation_classes.index(name) for name in target_class_names]

In [ ]:
# Generate prediction
with torch.no_grad():
    output = seg_model(input_tensor)['out'][0]

In [ ]:
# Obtener la clase predicha para cada píxel buscando la clase con la puntuación más alta.
output_predictions = output.argmax(0)

# Crear una máscara booleana separada para cada una de tus clases objetivo.
individual_masks = [(output_predictions == i) for i in class_indices]

# Apilar las máscaras individuales en un solo tensor de forma (num_masks, H, W).
stacked_masks = torch.stack(individual_masks, dim=0)

# Aplicar las máscaras de segmentación usando el tensor stacked_masks.
result = vutils.draw_segmentation_masks(image=(original_image_tensor * 255).byte(),
                                        masks=stacked_masks,
                                        alpha=0.5,
                                        colors=seg_colors)

# Visualizar la máscara (siéntete libre de establecer un `figsize` diferente)
helper_utils.display_images(processed_image=result, figsize=(10, 10))

## Conclusión

Ahora has visto un flujo de trabajo completo y práctico para usar `torchvision` en la resolución de problemas comunes de visión por computadora. Este laboratorio demostró los pasos esenciales para seleccionar un modelo preentrenado, comprender sus capacidades, preparar tus datos y utilizar el modelo para la inferencia con el fin de obtener resultados tangibles. Viste de primera mano lo crítica que es la visualización, no solo para los resultados finales, sino también para comprender y validar cada paso del proceso.

La conclusión más significativa es la ventaja estratégica de aprovechar los modelos preentrenados. Al construir sobre el conocimiento de arquitecturas de vanguardia (state-of-the-art), estás aprovechando eficazmente miles de horas de computación e investigación. Esto te permite prototipar y desplegar rápidamente soluciones de alta precisión sin el costo prohibitivo y el tiempo que requiere el entrenamiento desde cero. Esto no es solo un atajo; es la forma estándar y eficiente de construir aplicaciones de visión potentes.

Dominar la inferencia con modelos preentrenados es la base esencial para cualquier profesional de la visión por computadora. Con estas habilidades, ahora estás perfectamente posicionado para el siguiente paso en tu camino: aprender a adaptar y personalizar estos potentes modelos para tus propios conjuntos de datos únicos y tareas especializadas a través de técnicas como el aprendizaje por transferencia (transfer learning) y el ajuste fino (fine-tuning).